In [1]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
import torch
from transformers import AutoTokenizer
from transformers.models.llama import modeling_llama
from lexplainet.heatmap import clean_tokens, pdf_heatmap, plot_text_heatmap, interactive_text_heatmap

/home/erfan/venvs/llm_pruning/lib/python3.10/site-packages/transformers/utils/hub.py:111: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [3]:
from lexplainet.implicit.rules import dropout_rule, identity_rule, epsilon_rule
from lexplainet.implicit.composite_core import patch_composite, undo_patch_all
from lexplainet.implicit.model_specific_patches import gated_mlp, layer_norm_forward, rms_norm_forward, patch_attention

In [4]:
path = 'meta-llama/Llama-3.2-1B-Instruct'
model = modeling_llama.LlamaForCausalLM.from_pretrained(
    path,
    device_map='cuda',
    torch_dtype=torch.bfloat16
)

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(path)

# Deactivate gradients on parameters
for param in model.parameters():
    param.requires_grad = False

# Optionally enable gradient checkpointing (2x forward pass)
model.train()
model.gradient_checkpointing_enable()



undo_patch_all(model)
from transformers.models.llama import modeling_llama
patch_attention(modeling_llama)

from transformers.models.llama.modeling_llama import LlamaMLP, LlamaRMSNorm
patch_composite(model, {
    LlamaMLP: (gated_mlp, {}),
})

composite = {
            torch.nn.Linear: (epsilon_rule, {}),
            torch.nn.SiLU: (identity_rule, {}),
            LlamaRMSNorm: (rms_norm_forward, {}),
            torch.nn.Dropout: (dropout_rule, {}),
}

# undo_patch_all(model)
patch_composite(model, composite)

In [5]:
module_to_capture = model.model.layers[1].mlp.down_proj

In [6]:
def forward_hook_wrapper(module):
    def forward_hook(module, input, output):
        print("Input shape:", input[0].shape)
        print("Output shape:", output.shape)   
        # save the activation and make sure the gradient is also saved in the .grad attribute after the backward pass
        module.input = input
        module.output = output
        module.output.retain_grad() if module.output.requires_grad else None
    return forward_hook

hook_handles = []
for name, module in model.named_modules():
    if module == module_to_capture:
        print(f"Registering forward hook on module: {name}")
        hook_handle = module.register_forward_hook(forward_hook_wrapper(module))
        hook_handles.append(hook_handle)

Registering forward hook on module: model.layers.1.mlp.down_proj


In [7]:
prompt = """Context: Mount Everest attracts many climbers, including highly experienced mountaineers.
There are two main climbing routes, one approaching the summit from the southeast in Nepal (known as the standard route)
and the other from the north in Tibet. While not posing substantial technical climbing challenges on the standard route,
Everest presents dangers such as altitude sickness, weather, and wind, as well as hazards from avalanches and the Khumbu Icefall.
As of November 2022, 310 people have died on Everest. Over 200 bodies remain on the mountain and have not been removed
due to the dangerous conditions. The first recorded efforts to reach Everest's summit were made by British mountaineers.
As Nepal did not allow foreigners to enter the country at the time, the British made several attempts on the north ridge route
from the Tibetan side. After the first reconnaissance expedition by the British in 1921 reached 7,000 m (22,970 ft) on the
North Col, the 1922 expedition pushed the north ridge route up to 8,320 m (27,300 ft), marking the first time a human had
climbed above 8,000 m (26,247 ft). The 1924 expedition resulted in one of the greatest mysteries on Everest to this day:
George Mallory and Andrew Irvine made a final summit attempt on 8 June but never returned, sparking debate as to whether
they were the first to reach the top. Tenzing Norgay and Edmund Hillary made the first documented ascent of Everest in 1953,
using the southeast ridge route. Norgay had reached 8,595 m (28,199 ft) the previous year as a member of the 1952 Swiss expedition.
The Chinese mountaineering team of Wang Fuzhou, Gonpo, and Qu Yinhua made the first reported ascent of the peak from the
north ridge on 25 May 1960.
Question: How high did they climb in 1922? According to the text, the 1922 expedition reached 8,"""
# Get input embeddings
input_ids = tokenizer(prompt, return_tensors="pt", add_special_tokens=True).input_ids.to(model.device)
input_embeds = model.get_input_embeddings()(input_ids)

In [8]:
output_logits = model(inputs_embeds=input_embeds.requires_grad_(), use_cache=False).logits
# Take the maximum logit at last token position. You can also explain any other token, or several tokens together!
max_logits, max_indices = torch.max(output_logits[0, -1, :], dim=-1)

# Backward pass (the relevance is initialized with the value of max_logits)
max_logits.backward(retain_graph=True)
# torch.autograd.grad(outputs=max_logits, inputs=input_embeds, grad_outputs=torch.ones_like(max_logits), retain_graph=True)  # Retain graph if you want to do multiple backward passes for different tokens.

# Obtain relevance. (Works at any layer in the model!)
relevance = (input_embeds.grad * input_embeds).float().sum(-1).cpu()  # Cast to float32 for higher precision

# Normalize relevance between [-1, 1]
relevance = relevance / relevance.abs().max()

# Remove special characters that are not compatible wiht LaTeX
tokens = tokenizer.convert_ids_to_tokens(input_ids[0])
tokens = clean_tokens(tokens)

Input shape: torch.Size([1, 423, 8192])
Output shape: torch.Size([1, 423, 2048])
Input shape: torch.Size([1, 423, 8192])
Output shape: torch.Size([1, 423, 2048])


In [9]:
# This can be used to compute relevances at weight scale!
def weight_relevance(input, weight, out_grad):
    """
    inp:      [B, T, in_features]
    weight:   [out_features, in_features]
    out_grad: [B, T, out_features]

    returns:
        contrib: [B, T, out_features, in_features]
    """
    assert input.ndim == 3
    assert weight.ndim == 2
    assert out_grad.ndim == 3

    B, T, in_features = input.shape
    out_features, weight_in_features = weight.shape

    assert weight_in_features == in_features
    assert out_grad.shape == (B, T, out_features)

    return (
        input.unsqueeze(2)        # [B, T, 1, in_features]
        * weight.view(1, 1, out_features, in_features)
        * out_grad.unsqueeze(-1) # [B, T, out_features, 1]
    )

In [10]:
weight_relevance = weight_relevance(module_to_capture.input[0], module_to_capture.weight, module_to_capture.output.grad)

In [11]:
# assertion
# our assertion states that sum of weight relevances at each row should be equal to output or neuron relevances!
out_rel = (module_to_capture.output.grad * module_to_capture.output).sum(dim=(0, 1)).float()

w_rel_sum = weight_relevance.float().sum(dim=(0, 1, -1))
w_rel_sum_norm = w_rel_sum / w_rel_sum.norm()
out_rel_norm = out_rel / out_rel.norm()

cosine_sim= (w_rel_sum_norm * out_rel_norm).sum()
cosine_sim

tensor(1.0000, device='cuda:0', grad_fn=<SumBackward0>)

In [12]:
# remove hooks
for handle in hook_handles:
    handle.remove()